# DMS v6 — bản gọn, có chú thích từng cell

**18 cell code:** 14 bước chính và 4 cell xuất file/đóng gói Raspberry Pi.
Mỗi cell có mục đích, đầu ra và chú thích tiếng Việt. Các hàm liên quan được gộp chung
theo công việc; cell có `def` chỉ chuẩn bị hàm, các cell chạy dữ liệu/train được ghi rõ.

**Chạy:** upload lên Colab → chọn T4 GPU → Run all. Không cần tự tạo/upload CSV.
Chỉ cần chỉnh cấu hình ở **cell 2**. Lần đầu cần mạng để tải thư viện, dữ liệu và weights.
CPU dùng được để chuẩn bị dữ liệu và chạy thử `quick_check`; train đầy đủ cần T4.

| Nhánh | Dữ liệu tự tải | Nhãn |
|---|---|---|
| Hành vi | State Farm | distracted / normal / phone |
| Buồn ngủ | DDD | alert / drowsy |
| Ngáp | Yawn Eye Dataset New, chỉ lấy yawn/no_yawn | not_yawn / yawn |

Đây là **ba bộ dữ liệu, ba model độc lập**. Ngáp không đồng nghĩa buồn ngủ.
State Farm chia theo tài xế; DDD/ngáp thiếu ID nên chỉ chia theo cụm ảnh/ROI,
chưa chứng minh độc lập theo người. Cần đánh giá thêm trên người và camera mới.
Code nằm trực tiếp trong cell; không có source ẩn, base64 hay `exec`.


## Cell 1. Cài thư viện

**Làm gì:** Cài các package cần cho xử lý ảnh, train và xuất ONNX.

**Đầu ra:** Môi trường có đủ thư viện.

In [ ]:
# CELL 1: CÀI THƯ VIỆN
# Mục đích: Cài các package cần cho xử lý ảnh, train và xuất ONNX.
# Kết quả: Môi trường có đủ thư viện.


# 1. Cài các thư viện công khai (không cài helper riêng)
%pip install -q "ultralytics==8.4.49" "onnx>=1.16,<2" "onnxruntime>=1.18,<2" kagglehub pandas scipy scikit-learn matplotlib requests tqdm

## Cell 2. Cấu hình thí nghiệm

**Làm gì:** Import thư viện, đặt nhãn, đường dẫn, seed và các cấu hình cần train.

**Đầu ra:** WORK, TASKS, EXPERIMENTS và config.json.

**Cell cần chỉnh:** `PROFILE`, `EPOCHS`, `SAVE_TO_DRIVE`. `balanced` mặc định; `quick_check` chỉ chạy thử tối đa 2 epochs. Bật lưu Drive trước khi Run all nếu muốn sao lưu.

In [ ]:
# CELL 2: CẤU HÌNH THÍ NGHIỆM
# Mục đích: Import thư viện, đặt nhãn, đường dẫn, seed và các cấu hình cần train.
# Kết quả: WORK, TASKS, EXPERIMENTS và config.json.


import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
BASE = Path('/content') if IN_COLAB else Path.cwd()
RUNTIME = BASE / 'dms_v6_pi_files'
RUNTIME.mkdir(parents=True, exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(RUNTIME / '.yolo_config')
import hashlib
import inspect
import json
import os
import random
import shutil
import time
from copy import copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.data.dataset import ClassificationDataset
from ultralytics.models.yolo.classify import ClassificationTrainer, ClassificationValidator
import re
import warnings
import cv2
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler
import io
import zipfile
import requests
import gzip
from pathlib import Path, PurePosixPath
import stat
import tarfile
from urllib.parse import urlparse

try:
    from IPython.display import display
except ImportError:
    display = print
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('Chưa có GPU: vẫn chuẩn bị dữ liệu được. Train đầy đủ cần bật T4 trước cell train.')
else:
    print('GPU:', torch.cuda.get_device_name(0))

# --- Nhãn và nguồn dữ liệu ---
TASKS = {
    'behavior': ['distracted', 'normal', 'phone'],
    'vigilance': ['alert', 'drowsy'],
    'yawn': ['not_yawn', 'yawn'],
}
CLASSES = list(TASKS['behavior'])
SF_MAP = {f'c{i}': ('normal' if i == 0 else 'phone' if i < 5 else 'distracted') for i in range(10)}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
ARCHIVE_EXTENSIONS = {'.tar', '.tgz', '.gz', '.zip', '.rar', '.7z'}
BASE_SIZE = 320
ACTIVE_TASK = None
ACTIVE_FRAMES = None
ACTIVE_SEGMENTS = None
DATASETS = {
    'state_farm': ('rightway11/state-farm-distracted-driver-detection', 1),
    'ddd': ('ismailnasri20/driver-drowsiness-dataset-ddd', 1),
    'yawn_eye': ('serenaraju/yawn-eye-dataset-new', 1),
}
MODEL_NAME = 'face_detection_yunet_2023mar.onnx'
MODEL_SHA256 = '8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4'

# Lưu cấu hình hoặc kết quả thành JSON để xem lại sau khi train.
def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

# Cố định seed cho Python, NumPy và PyTorch.
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Tính checksum để kiểm tra đúng file model.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

# Đưa ảnh về khung vuông, giữ tỉ lệ và thêm padding.
def square_image(image, size=BASE_SIZE):
    """Identical first step for materialized training images and Pi inference."""
    image = ImageOps.exif_transpose(image).convert('RGB')
    return ImageOps.pad(image, (size, size), method=Image.Resampling.BILINEAR, color=(114, 114, 114), centering=(0.5, 0.5))

# Liệt kê các file ảnh trong thư mục dữ liệu.
def image_files(root):
    return sorted((p.resolve() for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS))

# Đặt nhãn và dữ liệu đang dùng cho một trong ba nhánh.
def activate(task, frames=None, segments=None):
    global CLASSES
    'Training is sequential on one GPU; task-specific labels are explicit.'
    global ACTIVE_TASK, ACTIVE_FRAMES, ACTIVE_SEGMENTS
    CLASSES = list(TASKS[task])
    ACTIVE_TASK, ACTIVE_FRAMES, ACTIVE_SEGMENTS = (task, frames, segments)

# --- CẤU HÌNH CẦN CHỈNH KHI CHẠY ---
PROFILE = "balanced" # @param ["quick_check", "balanced", "thorough"]
EPOCHS = 50 # @param {type:"integer"}
SEED = 42
NUM_WORKERS = 2
SAVE_TO_DRIVE = False # @param {type:"boolean"}
DRIVE_RESULTS_FOLDER = '/content/drive/MyDrive/dms_results'
STATE_FARM_EXISTING_FOLDER = ''
DDD_EXISTING_FOLDER = ''
YAWN_EXISTING_FOLDER = ''
if EPOCHS < 1 or NUM_WORKERS < 0:
    raise ValueError('EPOCHS >= 1 và NUM_WORKERS >= 0.')
if PROFILE not in {'quick_check', 'balanced', 'thorough'}:
    raise ValueError('PROFILE không hợp lệ.')
if SAVE_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError('Drive mount chỉ dùng trên Google Colab.')
    from google.colab import drive
    drive.mount('/content/drive')
from datetime import datetime
from zoneinfo import ZoneInfo
RUN_NAME = 'dms_v6_' + datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).strftime('%Y%m%d_%H%M%S_%f')
WORK = BASE / RUN_NAME
WORK.mkdir(parents=True, exist_ok=False)
CACHE = BASE / '.cache' / 'dms_v6_data'
BACKUP = Path(DRIVE_RESULTS_FOLDER) / RUN_NAME if SAVE_TO_DRIVE else None
seed_everything(SEED)

def backup_results():
    if BACKUP is None:
        return
    for path in WORK.rglob('*'):
        if not path.is_file():
            continue
        relative = path.relative_to(WORK)
        if any((s in {'dataset', 'face_frames', '__pycache__', 'runtime'} for s in relative.parts)):
            continue
        if path.suffix.lower() not in {'.pt', '.onnx', '.json', '.csv', '.yaml', '.txt', '.zip'}:
            continue
        if 'runs' in relative.parts and path.suffix not in {'.csv', '.yaml'}:
            continue
        destination = BACKUP / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)
    print('Đã sao lưu kết quả:', BACKUP)
EXPERIMENTS = {'behavior': [dict(name='nano224', model='yolo11n-cls.pt', imgsz=224, batch=32), dict(name='small224', model='yolo11s-cls.pt', imgsz=224, batch=24)], 'vigilance': [dict(name='nano160', model='yolo11n-cls.pt', imgsz=160, batch=64)], 'yawn': [dict(name='nano160', model='yolo11n-cls.pt', imgsz=160, batch=32), dict(name='nano224', model='yolo11n-cls.pt', imgsz=224, batch=32)]}
if PROFILE == 'thorough':
    EXPERIMENTS['behavior'].append(dict(name='small320', model='yolo11s-cls.pt', imgsz=320, batch=12))
    EXPERIMENTS['vigilance'].append(dict(name='nano224', model='yolo11n-cls.pt', imgsz=224, batch=48))
if PROFILE == 'quick_check':
    EPOCHS = min(EPOCHS, 2)
    EXPERIMENTS = {task: candidates[:1] for task, candidates in EXPERIMENTS.items()}
write_json(WORK / 'config.json', dict(profile=PROFILE, seed=SEED, epochs=EPOCHS, experiments=EXPERIMENTS, datasets=DATASETS, selection='Validation image macro-F1, including missing face ROI as unknown for face tasks.'))
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True).stdout
(WORK / 'requirements_resolved.txt').write_text(freeze, encoding='utf-8')
print('Profile:', PROFILE, '| số ứng viên:', sum(map(len, EXPERIMENTS.values())))
print('Kết quả:', WORK, '| backup:', BACKUP)
print('Runtime trống: %.1f GB' % (shutil.disk_usage(BASE).free / 1024 ** 3))

## Cell 3. Chuẩn bị cách tải dữ liệu

**Làm gì:** Khai báo các hàm tải tiếp, giải nén và tự lấy metadata State Farm.

**Đầu ra:** Các hàm tải dữ liệu; cell này chưa tải bộ ảnh.

In [ ]:
# CELL 3: CHUẨN BỊ CÁCH TẢI DỮ LIỆU
# Mục đích: Khai báo các hàm tải tiếp, giải nén và tự lấy metadata State Farm.
# Kết quả: Các hàm tải dữ liệu; cell này chưa tải bộ ảnh.


# Kiểm tra file giải nén luôn nằm trong thư mục đích.
def safe_target(root, member):
    normalized = str(member).replace('\\', '/')
    path = PurePosixPath(normalized)
    if path.is_absolute() or '..' in path.parts or any((':' in p for p in path.parts)):
        raise ValueError(f'Unsafe archive path: {member}')
    root = Path(root).resolve()
    target = (root / Path(*path.parts)).resolve()
    if not target.is_relative_to(root):
        raise ValueError(f'Archive path escapes destination: {member}')
    return target

# Ghi một file trong archive; bỏ qua file đã giải nén hoàn chỉnh.
def _copy_member(stream, target, expected_size=None):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and expected_size is not None and (target.stat().st_size == expected_size):
        return
    part = target.with_name(target.name + '.extracting')
    with open(part, 'wb') as out:
        shutil.copyfileobj(stream, out, length=1024 * 1024)
    if expected_size is not None and part.stat().st_size != expected_size:
        raise IOError(f'Incomplete extraction: {target.name}')
    os.replace(part, target)

# Giải nén dữ liệu, kiểm tra đường dẫn và file lồng nhau.
def extract_archive(archive, destination, scratch, max_depth=3):
    """Unpack tar→rar (YawDD), zip, tar.gz, or ordinary rar/7z without executing files."""
    archive, destination, scratch = (Path(archive), Path(destination), Path(scratch))
    destination.mkdir(parents=True, exist_ok=True)
    scratch.mkdir(parents=True, exist_ok=True)
    nested = []

    def output_path(name):
        if Path(str(name).replace('\\', '/')).suffix.lower() in ARCHIVE_EXTENSIONS:
            target = safe_target(scratch / 'nested', name)
            nested.append(target)
            return target
        return safe_target(destination, name)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                safe_target(destination, entry.filename)
                if stat.S_ISLNK(entry.external_attr >> 16):
                    raise ValueError('Archive symbolic links are not accepted.')
                if entry.is_dir():
                    continue
                with z.open(entry) as stream:
                    _copy_member(stream, output_path(entry.filename), entry.file_size)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive, 'r:*') as tf:
            for entry in tf:
                safe_target(destination, entry.name)
                if entry.isdir():
                    continue
                if not entry.isfile():
                    raise ValueError('Archive links/special files are not accepted.')
                with tf.extractfile(entry) as stream:
                    _copy_member(stream, output_path(entry.name), entry.size)
    else:
        with archive.open('rb') as f:
            magic = f.read(8)
        if magic[:2] == b'\x1f\x8b':
            inner = scratch / 'gzip_payload'
            with gzip.open(archive, 'rb') as stream:
                _copy_member(stream, inner)
            nested.append(inner)
        else:
            try:
                import libarchive
            except (ImportError, OSError) as e:
                raise RuntimeError('RAR extraction needs pip install libarchive-c and system libarchive.') from e
            with libarchive.file_reader(str(archive)) as entries:
                for entry in entries:
                    safe_target(destination, entry.pathname)
                    if entry.isdir:
                        continue
                    if entry.issym or entry.islnk or (not entry.isfile):
                        raise ValueError('Archive links/special files are not accepted.')
                    target = output_path(entry.pathname)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    expected = entry.size
                    if target.is_file() and expected is not None and (target.stat().st_size == expected):
                        continue
                    part = target.with_name(target.name + '.extracting')
                    with part.open('wb') as out:
                        for block in entry.get_blocks():
                            out.write(block)
                    if expected is not None and part.stat().st_size != expected:
                        raise IOError(f'Incomplete RAR entry: {target.name}')
                    os.replace(part, target)
    if nested and max_depth <= 0:
        raise ValueError('Archive nesting is deeper than expected.')
    for child in nested:
        extract_archive(child, destination, scratch / ('level_' + str(max_depth)), max_depth - 1)

# Tải archive qua HTTP, hỗ trợ tải tiếp và kiểm tra kích thước.
def download_archive(url, cache, source, expected_bytes=None):
    """Stream and resume supported HTTP downloads; never store signed URLs in receipts."""
    parsed = urlparse(url)
    if parsed.scheme not in {'http', 'https'} or not parsed.netloc:
        raise ValueError('Provide an HTTP(S) archive download URL.')
    cache = Path(cache)
    cache.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(url.encode()).hexdigest()
    target = cache / f'{source}_{digest[:16]}.archive'
    receipt = target.with_suffix('.download.json')
    if target.is_file() and receipt.is_file():
        info = json.loads(receipt.read_text(encoding='utf-8'))
        if info['bytes'] == target.stat().st_size and info['url_sha256'] == digest:
            if expected_bytes is None or info['bytes'] == expected_bytes:
                return target
    part = target.with_suffix('.part')
    if parsed.netloc.lower() in {'drive.google.com', 'docs.google.com'}:
        if '/folders/' in parsed.path:
            raise ValueError('Use a direct archive file link, not a Google Drive folder link.')
        import gdown
        result = gdown.download(url=url, output=str(part), fuzzy=True, resume=True, quiet=False)
        if result is None:
            raise RuntimeError('Drive archive could not be downloaded; check access or quota.')
        total = part.stat().st_size
    else:
        offset = part.stat().st_size if part.is_file() else 0
        headers = {'User-Agent': 'DMS-Colab-Downloader/1.0'}
        if offset:
            headers['Range'] = f'bytes={offset}-'
        with requests.get(url, headers=headers, stream=True, timeout=(20, 90)) as response:
            if response.status_code not in {200, 206}:
                raise RuntimeError(f'Download failed: HTTP {response.status_code} from {parsed.netloc}')
            if 'text/html' in response.headers.get('Content-Type', '').lower():
                raise ValueError('Received an HTML/login page; provide the actual archive download URL.')
            if response.status_code == 206:
                content_range = response.headers.get('Content-Range', '')
                if not content_range.startswith(f'bytes {offset}-'):
                    raise ValueError('Server returned an unexpected resume range.')
            else:
                offset = 0
            length = int(response.headers.get('Content-Length', 0))
            total = offset + length if length else None
            if expected_bytes is not None and total is not None and (total != expected_bytes):
                raise ValueError('Download size differs from the verified source size.')
            needed = max(0, (total or expected_bytes or 0) - offset)
            if needed and shutil.disk_usage(cache).free < needed + 512 * 1024 ** 2:
                raise OSError('Not enough free runtime disk space for this archive.')
            with part.open('ab' if offset else 'wb') as out, tqdm(total=total, initial=offset, unit='B', unit_scale=True, desc=source) as bar:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        out.write(chunk)
                        bar.update(len(chunk))
        if total is not None and part.stat().st_size != total:
            raise IOError('Download incomplete; rerun this cell to resume.')
        total = part.stat().st_size
    if expected_bytes is not None and total != expected_bytes:
        raise IOError('Download does not match the expected file size.')
    os.replace(part, target)
    receipt.write_text(json.dumps(dict(url_sha256=digest, host=parsed.netloc, bytes=total)), encoding='utf-8')
    return target

# Tự lấy bảng tài xế State Farm nếu bộ ảnh chưa có CSV.
def ensure_state_farm_metadata(root, session=requests):
    root = Path(root)
    existing = sorted(root.rglob('driver_imgs_list.csv'))
    if existing:
        return existing[0]
    url = 'https://www.kaggle.com/api/v1/datasets/download/rightway11/state-farm-distracted-driver-detection/driver_imgs_list.csv'
    response = session.get(url, timeout=(20, 60))
    response.raise_for_status()
    payload = response.content
    if payload.startswith(b'PK'):
        with zipfile.ZipFile(io.BytesIO(payload)) as z:
            names = [n for n in z.namelist() if Path(n).name == 'driver_imgs_list.csv']
            if len(names) != 1:
                raise ValueError('Metadata archive has no unique driver_imgs_list.csv.')
            payload = z.read(names[0])
    meta = pd.read_csv(io.BytesIO(payload), dtype=str)
    if not {'subject', 'classname', 'img'}.issubset(meta) or meta.empty:
        raise ValueError('Public State Farm metadata is invalid. Rerun the data-download cell.')
    if meta[['subject', 'classname', 'img']].isna().any().any() or meta.duplicated(['classname', 'img']).any():
        raise ValueError('State Farm metadata has empty or ambiguous identities.')
    root.mkdir(parents=True, exist_ok=True)
    target = root / 'driver_imgs_list.csv'
    target.write_bytes(payload)
    return target

# Lấy ba bộ dữ liệu từ cache, folder có sẵn hoặc Kaggle.
def acquire_datasets(cache, overrides=None):
    """Public version-pinned Kaggle mirrors; no Kaggle competition enrolment required."""
    cache = Path(cache)
    cache.mkdir(parents=True, exist_ok=True)
    roots, receipts = ({}, [])
    for source, (handle, version) in DATASETS.items():
        local = str((overrides or {}).get(source, '')).strip()
        if local:
            root = Path(local).expanduser()
            if not root.is_dir() or not image_files(root):
                raise FileNotFoundError(f'{source}: configured local directory has no images: {root}')
            method = 'existing_local'
        else:
            try:
                import kagglehub
                root = Path(kagglehub.dataset_download(f'{handle}/versions/{version}'))
                method = 'kagglehub'
            except Exception as e:
                print(f'{source}: public ZIP fallback ({type(e).__name__}).')
                url = f'https://www.kaggle.com/api/v1/datasets/download/{handle}?datasetVersionNumber={version}'
                archive = download_archive(url, cache / 'archives', source)
                root = cache / 'extracted' / source
                extract_archive(archive, root, cache / 'extract_scratch' / source)
                method = 'public_zip'
        if source == 'state_farm':
            ensure_state_farm_metadata(root)
        roots[source] = root.resolve()
        receipts.append(dict(source=source, handle=handle, version=version, method=method, version_verified=method != 'existing_local', folder=str(roots[source])))
        print(f'{source}: {roots[source]}')
    return (roots, receipts)

## Cell 4. Tải ba bộ và đọc nhãn

**Làm gì:** Tải hoặc dùng cache, ghép ảnh với nhãn và thống kê số ảnh từng lớp.

**Đầu ra:** ROOTS, tables, downloads.json và các bảng auto_labels.csv.

Không cần tự viết CSV. State Farm dùng ID tài xế thật; DDD/ngáp không tự đoán ID từ tên ảnh. Ảnh mắt Open/Closed không được dùng làm nhãn ngáp.

In [ ]:
# CELL 4: TẢI BA BỘ VÀ ĐỌC NHÃN
# Mục đích: Tải hoặc dùng cache, ghép ảnh với nhãn và thống kê số ảnh từng lớp.
# Kết quả: ROOTS, tables, downloads.json và các bảng auto_labels.csv.


# Ghép ảnh State Farm với nhãn và ID tài xế thật.
def index_state_farm(root):
    root = Path(root)
    paths = sorted(root.rglob('driver_imgs_list.csv'))
    if not paths:
        raise FileNotFoundError('State Farm requires driver_imgs_list.csv from the original competition.')
    metadata = pd.read_csv(paths[0], dtype=str)
    if not {'img', 'classname', 'subject'}.issubset(metadata):
        raise ValueError('Invalid State Farm metadata.')
    subjects = {(r.classname, r.img): r.subject for r in metadata.itertuples()}
    rows = []
    for path in image_files(root):
        tag = next((p for p in reversed(path.parts[:-1]) if p in SF_MAP), None)
        if tag is None:
            continue
        subject = subjects.get((tag, path.name))
        if not subject:
            raise ValueError(f'Missing State Farm driver ID: {path}')
        rows.append(dict(filepath=str(path), label=SF_MAP[tag], subject_id=subject, source='state_farm', task='behavior', subtype=tag))
    if not rows:
        raise ValueError('No State Farm class folders c0...c9 found.')
    return pd.DataFrame(rows)

# Chuẩn hóa tên thư mục Drowsy/Non Drowsy thành nhãn thống nhất.
def ddd_label(value):
    key = re.sub('[^a-z]', '', str(value).lower())
    return {'alert': 'alert', 'awake': 'alert', 'nondrowsy': 'alert', 'notdrowsy': 'alert', 'drowsy': 'drowsy'}.get(key, str(value))

# Đọc ảnh DDD; giữ trống ID người/video khi chưa có metadata.
def index_ddd(root, metadata_csv=None):
    """Read provided face images; never manufacture subject IDs from filenames."""
    root = Path(root).resolve()
    rows = []
    for path in image_files(root):
        label = next((ddd_label(part) for part in reversed(path.relative_to(root).parts[:-1]) if ddd_label(part) in TASKS['vigilance']), None)
        if label:
            rows.append(dict(filepath=str(path), label=label, subject_id='', video_id=''))
    frame = pd.DataFrame(rows, columns=['filepath', 'label', 'subject_id', 'video_id'])
    if metadata_csv:
        meta = pd.read_csv(metadata_csv, dtype=str).fillna('')
        if not {'filepath', 'label'}.issubset(meta):
            raise ValueError('DDD metadata needs filepath,label and optional subject_id,video_id.')
        for col in ['subject_id', 'video_id']:
            if col not in meta:
                meta[col] = ''
            meta[col] = meta[col].str.strip()
        meta['filepath'] = meta.filepath.map(lambda x: str((root / x).resolve()))
        meta['label'] = meta.label.map(ddd_label)
        if not set(meta.label).issubset(TASKS['vigilance']):
            raise ValueError('DDD labels must be alert/non-drowsy or drowsy; numeric codes are not inferred.')
        if meta.duplicated('filepath').any():
            raise ValueError('Repeated paths in DDD metadata.')
        for r in meta.itertuples():
            if not Path(r.filepath).is_file():
                raise FileNotFoundError(r.filepath)
        if len(frame):
            existing = frame.set_index('filepath').label
            for r in meta.itertuples():
                if r.filepath in existing and existing[r.filepath] != r.label:
                    raise ValueError(f'DDD folder/metadata label conflict: {r.filepath}')
        frame = pd.concat([frame, meta], ignore_index=True).drop_duplicates('filepath', keep='last')
    if frame.empty:
        raise ValueError('DDD requires Drowsy/ and Non Drowsy/ image folders, or a ddd_manifest.csv.')
    frame['source'], frame['task'], frame['subtype'] = ('ddd', 'vigilance', frame.label)
    return frame.sort_values('filepath').reset_index(drop=True)

# Đọc hai lớp yawn/no_yawn và bỏ qua các thư mục ảnh mắt.
def index_yawn_images(root):
    rows = []
    for path in image_files(root):
        parts = [s.lower() for s in path.relative_to(Path(root).resolve()).parts[:-1]]
        label = next((s for s in reversed(parts) if s in {'yawn', 'no_yawn'}), None)
        if label is None:
            continue
        rows.append(dict(filepath=str(path), label='not_yawn' if label == 'no_yawn' else 'yawn', subject_id='', video_id='', source='yawn_eye', task='yawn', subtype=label, original_partition=next((s for s in parts if s in {'train', 'test', 'val'}), 'unspecified')))
    result = pd.DataFrame(rows)
    if result.empty or set(result.label) != set(TASKS['yawn']):
        raise ValueError('Yawn image source must have both yawn/ and no_yawn/ folders.')
    return result

# Tạo bảng ảnh/nhãn cho ba nhánh và kiểm tra đủ lớp.
def index_sources(roots):
    tables = {'behavior': index_state_farm(roots['state_farm']), 'vigilance': index_ddd(roots['ddd']), 'yawn': index_yawn_images(roots['yawn_eye'])}
    for task, table in tables.items():
        if set(table.label) != set(TASKS[task]):
            raise ValueError(f'{task}: missing expected labels.')
        tables[task] = table.fillna('')
    return tables

ROOTS, download_receipts = acquire_datasets(CACHE, overrides={'state_farm': STATE_FARM_EXISTING_FOLDER, 'ddd': DDD_EXISTING_FOLDER, 'yawn_eye': YAWN_EXISTING_FOLDER})
write_json(WORK / 'downloads.json', download_receipts)
tables = index_sources(ROOTS)
for task, table in tables.items():
    print(task, '|', len(table), 'ảnh | nguồn:', sorted(table.source.unique()))
    display(table.label.value_counts().rename('images').to_frame())
    table.to_csv(WORK / f'{task}_auto_labels.csv', index=False)
print('Đã tự đọc nhãn và metadata; không có bước upload CSV.')
backup_results()

## Cell 5. Chuẩn bị bộ phát hiện mặt

**Làm gì:** Tải YuNet, kiểm tra checksum và khai báo cách cắt mặt dùng chung với Pi.

**Đầu ra:** Weights YuNet, license và FaceCropper.

Chưa cắt cả dataset ở cell này. Ảnh không thấy mặt sẽ được ghi là unknown khi đánh giá.

In [ ]:
# CELL 5: CHUẨN BỊ BỘ PHÁT HIỆN MẶT
# Mục đích: Tải YuNet, kiểm tra checksum và khai báo cách cắt mặt dùng chung với Pi.
# Kết quả: Weights YuNet, license và FaceCropper.


# Tải YuNet chính thức và kiểm tra checksum.
def download_face_detector():
    destination = RUNTIME / MODEL_NAME
    if not destination.is_file() or sha256_file(destination) != MODEL_SHA256:
        url = ('https://github.com/opencv/opencv_zoo/raw/main/models/'
               'face_detection_yunet/face_detection_yunet_2023mar.onnx')
        response = requests.get(url, timeout=(20, 60))
        response.raise_for_status()
        if hashlib.sha256(response.content).hexdigest() != MODEL_SHA256:
            raise ValueError('Checksum YuNet không khớp nguồn đã kiểm tra.')
        destination.write_bytes(response.content)
    return destination

YUNET_LICENSE = """
MIT License

Copyright (c) 2020 Shiqi Yu <shiqi.yu@gmail.com>

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE."""

(RUNTIME / 'YUNET_LICENSE.txt').write_text(YUNET_LICENSE, encoding='utf-8')
download_face_detector()
print('YuNet:', RUNTIME / MODEL_NAME)

# Phát hiện mặt lớn nhất bằng YuNet và cắt ROI có viền 15%.
class FaceCropper:

    def __init__(self):
        model = RUNTIME / MODEL_NAME
        if not model.is_file() or hashlib.sha256(model.read_bytes()).hexdigest() != MODEL_SHA256:
            raise RuntimeError('Missing/mismatched YuNet weights. Rerun notebook setup or use the complete Pi bundle.')
        self.detector = cv2.FaceDetectorYN.create(str(model), '', (320, 320), 0.7, 0.3, 5000)

    def __call__(self, bgr):
        h, w = bgr.shape[:2]
        scale = 320 / max(h, w)
        sw, sh = (max(1, round(w * scale)), max(1, round(h * scale)))
        small = cv2.resize(bgr, (sw, sh))
        left_pad, top_pad = ((320 - sw) // 2, (320 - sh) // 2)
        square = cv2.copyMakeBorder(small, top_pad, 320 - sh - top_pad, left_pad, 320 - sw - left_pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
        _, faces = self.detector.detect(square)
        if faces is None or not len(faces):
            return None
        x, y, fw, fh = max(faces, key=lambda box: box[2] * box[3])[:4]
        x, y = ((x - left_pad) / scale, (y - top_pad) / scale)
        fw, fh = (fw / scale, fh / scale)
        pad_x, pad_y = (fw * 0.15, fh * 0.15)
        left, top = (max(0, int(x - pad_x)), max(0, int(y - pad_y)))
        right, bottom = (min(w, int(x + fw + pad_x)), min(h, int(y + fh + pad_y)))
        return bgr[top:bottom, left:right].copy() if right > left and bottom > top else None

## Cell 6. Quy tắc audit và chia nhóm

**Làm gì:** Khai báo kiểm tra ảnh hỏng/trùng và cách chia train/val/test theo nhóm.

**Đầu ra:** Các hàm audit và grouped_split; chưa chạy trên toàn bộ dữ liệu.

Hash pixel tìm ảnh trùng chính xác; pHash tìm ảnh gần trùng. Gộp nhóm trước khi chia, không dùng điểm model để sửa split.

In [ ]:
# CELL 6: QUY TẮC AUDIT VÀ CHIA NHÓM
# Mục đích: Khai báo kiểm tra ảnh hỏng/trùng và cách chia train/val/test theo nhóm.
# Kết quả: Các hàm audit và grouped_split; chưa chạy trên toàn bộ dữ liệu.


# Nối những ảnh phải nằm cùng một nhóm dữ liệu.
class UnionFind:

    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, i):
        while i != self.parent[i]:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i

    def union(self, a, b):
        a, b = (self.find(a), self.find(b))
        if a != b:
            self.parent[max(a, b)] = min(a, b)

# Tìm pHash gần nhau để phát hiện ảnh gần trùng.
class BKTree:
    """Exact Hamming-radius search over 64-bit perceptual hashes."""

    def __init__(self):
        self.root = None

    def add(self, value, index):
        if self.root is None:
            self.root = (value, index, {})
            return
        node = self.root
        while True:
            d = (value ^ node[0]).bit_count()
            if d == 0:
                return
            if d not in node[2]:
                node[2][d] = (value, index, {})
                return
            node = node[2][d]

    def query(self, value, radius):
        stack = [self.root] if self.root else []
        found = []
        while stack:
            node = stack.pop()
            d = (value ^ node[0]).bit_count()
            if d <= radius:
                found.append(node[1])
            stack.extend((child for edge, child in node[2].items() if d - radius <= edge <= d + radius))
        return found

# Kiểm tra ảnh hỏng, nhãn xung đột và nối cụm ảnh trùng/gần trùng.
def audit_and_group(frame, audit_dir, phash_radius=2):
    """Group subject IDs and near duplicates BEFORE splitting; no label-score tuning."""
    from scipy.fft import dctn
    audit_dir = Path(audit_dir)
    audit_dir.mkdir(parents=True, exist_ok=True)
    valid, corrupt = ([], [])
    for r in tqdm(frame.to_dict('records'), desc='Audit images'):
        try:
            with Image.open(r['filepath']) as im:
                rgb = ImageOps.exif_transpose(im).convert('RGB')
                gray = np.asarray(rgb.convert('L').resize((32, 32), Image.Resampling.LANCZOS))
                low = dctn(gray.astype(float), type=2)[:8, :8]
                bits = (low > np.median(low)).ravel()
                phash = sum((int(bit) << 63 - j for j, bit in enumerate(bits)))
                r.update(width=rgb.width, height=rgb.height, pixel_hash=hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest(), phash=f'{phash:016x}')
            valid.append(r)
        except Exception as e:
            corrupt.append(dict(filepath=r['filepath'], error=str(e)))
    pd.DataFrame(corrupt, columns=['filepath', 'error']).to_csv(audit_dir / 'corrupt_images.csv', index=False)
    df = pd.DataFrame(valid)
    if df.empty:
        raise ValueError('No decodable images remain.')
    conflicts = df.groupby('pixel_hash').label.nunique()
    conflicts = conflicts[conflicts > 1].index
    if len(conflicts):
        df[df.pixel_hash.isin(conflicts)].to_csv(audit_dir / 'label_conflicts.csv', index=False)
        raise ValueError('Identical images have conflicting labels; review label_conflicts.csv.')
    if 'split' in df:
        crossing = df.groupby('pixel_hash').split.nunique().loc[lambda s: s > 1].index
        if len(crossing):
            df[df.pixel_hash.isin(crossing)].to_csv(audit_dir / 'exact_cross_split.csv', index=False)
            raise ValueError('Identical images cross holdout splits; inspect exact_cross_split.csv.')
    df['_known'] = (df.subject_id != '').astype(int)
    df = df.sort_values(['_known', 'filepath'], ascending=[False, True]).drop_duplicates('pixel_hash').drop(columns='_known').reset_index(drop=True)
    uf, owners, trees, pairs = (UnionFind(len(df)), {}, {}, [])
    for i, r in df.iterrows():
        if r.subject_id:
            key = (r.source, r.subject_id)
            if key in owners:
                uf.union(i, owners[key])
            else:
                owners[key] = i
        for column in ['video_id', 'split_group']:
            value = r.get(column, '')
            if pd.notna(value) and str(value).strip():
                key = (r.source, column, str(value))
                if key in owners:
                    uf.union(i, owners[key])
                else:
                    owners[key] = i
        tree = trees.setdefault(r.source, BKTree())
        value = int(r.phash, 16)
        for j in tree.query(value, phash_radius):
            uf.union(i, j)
            pairs.append((df.iloc[j].filepath, r.filepath, (value ^ int(df.iloc[j].phash, 16)).bit_count()))
        tree.add(value, i)
    df['group_id'] = [f'g{uf.find(i):06d}' for i in range(len(df))]
    df.to_csv(audit_dir / 'audited_manifest.csv', index=False)
    pd.DataFrame(pairs, columns=['image_a', 'image_b', 'phash_distance']).to_csv(audit_dir / 'near_duplicate_pairs.csv', index=False)
    pd.crosstab(df.label, df.source).to_csv(audit_dir / 'source_by_class.csv')
    return df

# Chia train/val/test theo nhóm, giữ đủ lớp ở mỗi tập.
def grouped_split(df, seed=42, test_size=0.15, val_size=0.15):
    """Choose a split by label counts only. Model results never influence the split."""
    if not 0 < test_size < 1 or not 0 < val_size < 1 - test_size:
        raise ValueError('Invalid split fractions.')
    target = np.array([1 - test_size - val_size, val_size, test_size])
    total = df.label.value_counts().reindex(CLASSES, fill_value=0).to_numpy()
    best = None
    for attempt in range(100):
        state = seed + attempt
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=state)
        try:
            dev_idx, test_idx = next(outer.split(df, groups=df.group_id))
            dev = df.iloc[dev_idx]
            inner = GroupShuffleSplit(n_splits=1, test_size=val_size / (1 - test_size), random_state=state + 1000)
            tr, va = next(inner.split(dev, groups=dev.group_id))
        except ValueError:
            continue
        indices = [dev_idx[tr], dev_idx[va], test_idx]
        counts = np.array([df.iloc[ix].label.value_counts().reindex(CLASSES, fill_value=0).to_numpy() for ix in indices])
        if (counts == 0).any():
            continue
        score = np.abs(counts / total[None, :] - target[:, None]).mean()
        if best is None or score < best[0]:
            best = (score, indices)
    if best is None:
        raise ValueError('Not enough independent groups for every class in all three splits. Inspect near_duplicate_pairs.csv and add subject/session metadata.')
    result = df.copy()
    result['split'] = ''
    for name, ix in zip(['train', 'val', 'test'], best[1]):
        result.loc[result.index[ix], 'split'] = name
    assert result.groupby('group_id').split.nunique().max() == 1
    assert result.groupby('pixel_hash').split.nunique().max() == 1
    known = result[result.subject_id != '']
    assert known.groupby(['source', 'subject_id']).split.nunique().max() <= 1 if len(known) else True
    return result

## Cell 7. Xử lý ảnh và tạo train/val/test

**Làm gì:** Cắt mặt, nối cụm ảnh gốc/ROI rồi tạo dữ liệu cho ba nhánh.

**Đầu ra:** task_frames, unit_info, source_split_manifest.csv, roi_coverage.csv và dataset_facts.json.

Cell xử lý dữ liệu chính, có thể mất thời gian. Hành vi dùng toàn ảnh; DDD/ngáp dùng mặt. Ảnh không có mặt vẫn ở bảng đánh giá. DDD/ngáp dùng phép chia nội bộ mới, không gọi là benchmark tách người.

In [ ]:
# CELL 7: XỬ LÝ ẢNH VÀ TẠO TRAIN/VAL/TEST
# Mục đích: Cắt mặt, nối cụm ảnh gốc/ROI rồi tạo dữ liệu cho ba nhánh.
# Kết quả: task_frames, unit_info, source_split_manifest.csv, roi_coverage.csv và dataset_facts.json.


# Cắt mặt cho cả DDD và ảnh ngáp; ghi lại ảnh mất mặt.
def extract_ddd_faces(images, output, cropper=None):
    """Standardize DDD's face crops with the SAME detector/ROI used on Pi."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    cropper = cropper or FaceCropper()
    frames, units = ([], [])
    for row in tqdm(images.to_dict('records'), desc='Face ROI'):
        key = 'ddd_' + row['pixel_hash']
        original = row['filepath']
        unit = {**row, 'segment_id': key, 'requested_frames': 1, 'valid_face_frames': 0, 'decode_failures': 0, 'no_face_frames': 0}
        try:
            with Image.open(original) as image:
                rgb = np.asarray(ImageOps.exif_transpose(image).convert('RGB'))
            face = cropper(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
            if face is None:
                unit['no_face_frames'] = 1
            else:
                dest = output / (key + '.png')
                rgb_face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
                Image.fromarray(rgb_face).save(dest)
                face_hash = hashlib.sha256(str((rgb_face.shape[1], rgb_face.shape[0])).encode() + rgb_face.tobytes()).hexdigest()
                frames.append({**row, 'segment_id': key, 'filepath': str(dest.resolve()), 'original_path': original})
                unit['face_pixel_hash'] = face_hash
                unit['valid_face_frames'] = 1
        except (OSError, ValueError, cv2.error):
            unit['decode_failures'] = 1
        units.append(unit)
    return (pd.DataFrame(frames), pd.DataFrame(units))

# Tạo thư mục train/val/test từ bảng chia tập đã kiểm tra.
def materialize_dataset(frame, output_root):
    """Use all audited images. Unique names prevent collisions across source folders."""
    output_root = Path(output_root)
    if output_root.exists() and any(output_root.iterdir()):
        raise FileExistsError(f'Use a new output folder; refusing to mix datasets: {output_root}')
    for split in ['train', 'val', 'test']:
        for label in CLASSES:
            (output_root / split / label).mkdir(parents=True, exist_ok=True)
    df = frame.copy()
    paths = []
    for r in df.itertuples():
        dest = output_root / r.split / r.label / f'{r.pixel_hash}.png'
        with Image.open(r.filepath) as im:
            square_image(im).save(dest)
        paths.append(str(dest.resolve()))
    df['prepared_path'] = paths
    df.to_csv(output_root.parent / 'split_manifest.csv', index=False)
    facts = dict(images=len(df), group_split=True, subject_independent=bool((df.subject_id != '').all()), missing_subject_ids=int((df.subject_id == '').sum()), source_confounded=bool(any((df[df.label == c].source.nunique() == 1 and df[df.source.isin(df[df.label == c].source)].label.nunique() == 1 for c in CLASSES))), near_duplicate_method='64-bit pHash, radius=2 by default; not a proof of subject independence', base_image_size=BASE_SIZE, counts=pd.crosstab(df.split, df.label).to_dict())
    write_json(output_root.parent / 'dataset_facts.json', facts)
    return (df, facts)

# Nối cụm ảnh gốc và cụm mặt trước khi quyết định split.
def _join_roi_groups(raw, face_frames, face_units, audit):
    """Join original-image and face-crop similarity BEFORE selecting any holdout."""
    if face_frames.empty:
        raise ValueError('No faces found. Inspect roi_coverage.csv before training.')
    face_frames = face_frames.copy()
    face_frames['split_group'] = face_frames.group_id
    faces = audit_and_group(face_frames, audit)
    raw = raw.reset_index(drop=True).copy()
    uf = UnionFind(len(raw))
    owners = {}
    crop_groups = faces.set_index('pixel_hash').group_id.to_dict()
    units = face_units.set_index('original_path')
    for i, row in raw.iterrows():
        group_keys = [('raw', row.group_id)]
        crop_hash = units.loc[row.filepath].get('face_pixel_hash', '')
        if pd.notna(crop_hash) and crop_hash in crop_groups:
            group_keys.append(('roi', crop_groups[crop_hash]))
        for key in group_keys:
            if key in owners:
                uf.union(i, owners[key])
            else:
                owners[key] = i
    raw['group_id'] = [f'joined_{uf.find(i):07d}' for i in range(len(raw))]
    raw['split_group'] = raw.group_id
    return (raw, faces, units.reset_index())

# Chạy chuẩn bị dữ liệu và xuất bảng ROI coverage/split cho ba nhánh.
def prepare(tables, work, seed=42, cropper=None):
    """Build all three labelled-image tasks and retain failed ROI inputs in image metrics."""
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    prepared, all_units, all_raw = ({}, [], [])
    for task, table in tables.items():
        activate(task)
        print(f'Preparing {task}: {len(table)} original images', flush=True)
        raw = audit_and_group(table, work / task / 'audit_originals')
        protocol = 'subject_holdout' if raw.subject_id.ne('').all() else 'perceptual_cluster_holdout_subjects_unverified'
        raw['evaluation_protocol'] = protocol
        if task == 'behavior':
            raw = grouped_split(raw, seed=seed)
            frame = raw
        else:
            frames, units = extract_ddd_faces(raw, work / 'face_frames' / task, cropper)
            units['original_path'] = units.filepath
            units.to_csv(work / task / 'roi_coverage_before_split.csv', index=False)
            raw, frame, units = _join_roi_groups(raw, frames, units, work / task / 'audit_faces')
            raw = grouped_split(raw, seed=seed)
            split = raw.set_index('filepath')[['split', 'group_id', 'split_group']]
            for column in ['split', 'group_id', 'split_group']:
                frame[column] = frame.original_path.map(split[column])
                units[column] = units.original_path.map(split[column])
            units['metric_unit'] = 'image'
            all_units.append(units)
        for part in ['train', 'val', 'test']:
            if set(frame[frame.split == part].label) != set(TASKS[task]):
                raise ValueError(f'{task}/{part}: insufficient face coverage or independent groups for all classes.')
        if frame.groupby('group_id').split.nunique().max() != 1:
            raise AssertionError('ROI group crosses holdouts.')
        data, facts = materialize_dataset(frame, work / task / 'dataset')
        write_json(work / task / 'dataset_facts.json', dict(**facts, task=task, classes=TASKS[task], sources=sorted(raw.source.unique()), original_images_after_dedup=len(raw), prepared_images=len(data), evaluation_protocol=protocol, label_granularity='image', roi='whole_frame' if task == 'behavior' else 'yunet_largest_face_with_15pct_margin', split_note='New internal grouped holdout; not the published dataset benchmark split.'))
        prepared[task] = data
        all_raw.append(raw)
    units = pd.concat(all_units, ignore_index=True).fillna('')
    units.to_csv(work / 'roi_coverage.csv', index=False)
    pd.concat(all_raw, ignore_index=True).to_csv(work / 'source_split_manifest.csv', index=False)
    used_sources = sorted(set((source for table in tables.values() for source in table.source.unique())))
    write_json(work / 'source_availability.json', dict(used_sources=used_sources, source_count=len(used_sources), label_mode='Provided image/folder labels; CSV manifests generated automatically.', excluded_optional_sources={'yawdd': 'Requires manually verified interval labels; not used.', 'samdd': 'Published RGB archive was invalid; not used.'}))
    return (prepared, units)

task_frames, unit_info = prepare(tables, WORK, seed=SEED)
for task, frame in task_frames.items():
    print(task, '| lớp:', TASKS[task])
    display(pd.crosstab(frame.split, frame.label))
    facts = json.loads((WORK / task / 'dataset_facts.json').read_text(encoding='utf-8'))
    print('Phép đánh giá:', facts['evaluation_protocol'])
display(unit_info.groupby(['task', 'split'])[['requested_frames', 'valid_face_frames', 'no_face_frames', 'decode_failures']].sum())
backup_results()

## Cell 8. Xem ảnh mẫu trước khi train

**Làm gì:** Hiển thị ảnh sau chuẩn hóa để kiểm tra nhãn và vùng mặt.

**Đầu ra:** Lưới ảnh mẫu của ba nhánh.

In [ ]:
# CELL 8: XEM ẢNH MẪU TRƯỚC KHI TRAIN
# Mục đích: Hiển thị ảnh sau chuẩn hóa để kiểm tra nhãn và vùng mặt.
# Kết quả: Lưới ảnh mẫu của ba nhánh.


import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for row, (task, frame) in enumerate(task_frames.items()):
    for col, label in enumerate(TASKS[task]):
        sample = frame[(frame.split == 'train') & (frame.label == label)].sample(1, random_state=SEED).iloc[0]
        with Image.open(sample.prepared_path) as im:
            axes[row, col].imshow(im)
        axes[row, col].set_title(task + ' / ' + label)
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Cell 9. Dataset và augmentation

**Làm gì:** Định nghĩa cách nạp ảnh, augmentation và Trainer/Validator cơ sở.

**Đầu ra:** Các class dùng khi train; cell này chưa huấn luyện model.

Chỉ train có augmentation. Validation giữ preprocessing xác định; checkpoint chọn theo macro-F1.

In [ ]:
# CELL 9: DATASET VÀ AUGMENTATION
# Mục đích: Định nghĩa cách nạp ảnh, augmentation và Trainer/Validator cơ sở.
# Kết quả: Các class dùng khi train; cell này chưa huấn luyện model.


# Áp dụng augmentation cho train và resize cố định khi đánh giá.
class DMSDataset(ClassificationDataset):

    def __init__(self, root, args, augment=False, prefix='', names=None):
        kw = dict(root=root, args=args, augment=augment, prefix=prefix)
        if 'names' in inspect.signature(ClassificationDataset).parameters:
            kw['names'] = names
        super().__init__(**kw)
        steps = [T.Resize((args.imgsz, args.imgsz), interpolation=T.InterpolationMode.BILINEAR)]
        if augment:
            steps += [T.RandomApply([T.ColorJitter(brightness=0.25, contrast=0.2, saturation=0.2, hue=0.015)], p=0.7), T.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.95, 1.05), fill=114), T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.1)]
        steps += [T.ToTensor()]
        if augment:
            steps += [T.RandomErasing(p=0.05, scale=(0.01, 0.04), value=0.45)]
        self.torch_transforms = T.Compose(steps)

# Tính macro-F1 dùng chọn checkpoint thay vì chỉ nhìn accuracy.
class DMSValidator(ClassificationValidator):

    def build_dataset(self, img_path):
        return DMSDataset(img_path, self.args, augment=False, prefix=self.args.split, names=getattr(self, 'names', None))

    def get_stats(self):
        stats = super().get_stats()
        y = torch.cat(self.targets).cpu().numpy().reshape(-1)
        p = torch.cat(self.pred).cpu().numpy()[:, 0]
        macro = float(f1_score(y, p, labels=list(range(len(CLASSES))), average='macro', zero_division=0))
        stats['metrics/macro_f1'] = macro
        stats['metrics/balanced_accuracy'] = float(balanced_accuracy_score(y, p))
        stats['fitness'] = macro
        return stats

# Tạo Dataset/DataLoader cho quá trình huấn luyện.
class DMSTrainer(ClassificationTrainer):

    def build_dataset(self, img_path, mode='train', batch=None):
        return DMSDataset(img_path, self.args, augment=mode == 'train', prefix=mode, names=self.data.get('names'))

    def get_dataloader(self, dataset_path, batch_size=16, rank=-1, mode='train'):
        if rank not in {-1, 0}:
            raise ValueError('This notebook supports one GPU, not distributed training.')
        dataset = self.build_dataset(dataset_path, mode)
        sampler = None
        generator = torch.Generator().manual_seed(self.args.seed)
        if mode == 'train':
            y = np.array([s[1] for s in dataset.samples])
            counts = np.bincount(y, minlength=len(CLASSES))
            if (counts == 0).any():
                raise ValueError('A training class is empty.')
            sampler = WeightedRandomSampler(torch.as_tensor(1 / counts[y], dtype=torch.double), num_samples=len(y), replacement=True, generator=generator)
        loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler, shuffle=False, num_workers=self.args.workers, pin_memory=self.device.type == 'cuda', generator=generator)
        if mode != 'train':
            self.model.transforms = dataset.torch_transforms
        return loader

    def get_validator(self):
        self.loss_names = ['loss']
        return DMSValidator(self.test_loader, self.save_dir, args=copy(self.args), _callbacks=self.callbacks)

## Cell 10. Cách suy luận và tính điểm

**Làm gì:** Khai báo suy luận PyTorch/ONNX và tính accuracy, macro-F1, confusion matrix.

**Đầu ra:** Các hàm dự đoán và báo cáo; chưa chấm test.

Ảnh mất mặt tính unknown trong mẫu số F1/accuracy, tránh chỉ báo cáo các ảnh dễ xử lý.

In [ ]:
# CELL 10: CÁCH SUY LUẬN VÀ TÍNH ĐIỂM
# Mục đích: Khai báo suy luận PyTorch/ONNX và tính accuracy, macro-F1, confusion matrix.
# Kết quả: Các hàm dự đoán và báo cáo; chưa chấm test.


# Đọc ảnh đánh giá theo đúng thứ tự và preprocessing.
class EvaluationImages(Dataset):

    def __init__(self, frame, size):
        self.frame = frame.reset_index(drop=True)
        self.transform = T.Compose([T.Resize((size, size), interpolation=T.InterpolationMode.BILINEAR), T.ToTensor()])

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        r = self.frame.iloc[i]
        with Image.open(r.prepared_path) as im:
            x = self.transform(im.convert('RGB'))
        return (x, CLASSES.index(r.label))

# Lấy xác suất từng lớp từ checkpoint PyTorch hoặc ONNX.
def predict_frame(model_path, frame, size, device='cpu', batch=32):
    """Tensor input bypasses the stock crop transform, including ONNX/TFLite."""
    wrapper = YOLO(str(model_path), task='classify')
    loader = DataLoader(EvaluationImages(frame, size), batch_size=batch, shuffle=False, num_workers=0)
    probabilities, labels = ([], [])
    if Path(model_path).suffix == '.pt':
        model = wrapper.model.to(device).float().eval()
        with torch.inference_mode():
            for x, y in loader:
                out = model(x.to(device))
                out = out[0] if isinstance(out, (tuple, list)) else out
                probabilities.append(out.float().cpu().numpy())
                labels.extend(y.tolist())
        names = model.names
    else:
        loader = DataLoader(EvaluationImages(frame, size), batch_size=1, num_workers=0)
        names = None
        for x, y in loader:
            result = wrapper.predict(x, device=device, imgsz=size, verbose=False)[0]
            probabilities.append(result.probs.data.cpu().numpy()[None, :])
            labels.extend(y.tolist())
            names = result.names
    assert [names[i] for i in range(len(CLASSES))] == CLASSES, 'Class order mismatch.'
    return (np.asarray(labels), np.concatenate(probabilities))

# Lưu accuracy, macro-F1, confusion matrix và thống kê lỗi.
def score_predictions(y, probabilities, frame, report_dir, bootstrap=400):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    p = probabilities.argmax(axis=1)
    summary = dict(accuracy=float(accuracy_score(y, p)), macro_f1=float(f1_score(y, p, labels=range(len(CLASSES)), average='macro', zero_division=0)), balanced_accuracy=float(balanced_accuracy_score(y, p)), n=len(y))
    report = classification_report(y, p, labels=list(range(len(CLASSES))), target_names=CLASSES, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(report_dir / 'classification_report.csv')
    cm = confusion_matrix(y, p, labels=list(range(len(CLASSES))))
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(report_dir / 'confusion_matrix.csv')
    results = frame.reset_index(drop=True).copy()
    results['prediction'] = [CLASSES[i] for i in p]
    results['confidence'] = probabilities.max(axis=1)
    for i, label in enumerate(CLASSES):
        results[f'p_{label}'] = probabilities[:, i]
    results.to_csv(report_dir / 'predictions.csv', index=False)
    results[results.label != results.prediction].to_csv(report_dir / 'errors.csv', index=False)
    if bootstrap and 'group_id' in results:
        groups = results.group_id.to_numpy()
        unique = np.unique(groups)
        rng = np.random.default_rng(2026)
        values = []
        members = {g: np.flatnonzero(groups == g) for g in unique}
        for _ in range(bootstrap):
            ix = np.concatenate([members[g] for g in rng.choice(unique, len(unique), replace=True)])
            if len(np.unique(y[ix])) == len(CLASSES):
                values.append(f1_score(y[ix], p[ix], labels=range(len(CLASSES)), average='macro', zero_division=0))
        if values:
            summary['macro_f1_group_bootstrap_95ci'] = np.quantile(values, [0.025, 0.975]).tolist()
            summary['bootstrap_valid_resamples'] = len(values)
    for col in ['source', 'subtype']:
        breakdown = []
        for key, sub in results.groupby(col):
            breakdown.append(dict(group=key, n=len(sub), accuracy=float((sub.label == sub.prediction).mean())))
        pd.DataFrame(breakdown).to_csv(report_dir / f'{col}_breakdown.csv', index=False)
    write_json(report_dir / 'summary.json', summary)
    return summary

# Tính điểm cả ảnh không có mặt bằng dự đoán unknown.
def unit_scores(frames, probabilities, segments, report_dir=None, min_face_coverage=0.3):
    """Evaluate DDD images or YawDD intervals, INCLUDING failed/no-face inputs."""
    labels = TASKS[ACTIVE_TASK]
    rows = []
    for seg in segments.to_dict('records'):
        positions = np.flatnonzero(frames.segment_id.to_numpy() == seg['segment_id'])
        if not len(positions) and pd.notna(seg.get('face_pixel_hash')):
            positions = np.flatnonzero(frames.pixel_hash.to_numpy() == seg['face_pixel_hash'])
        coverage = float(seg['valid_face_frames']) / max(int(seg['requested_frames']), 1)
        eligible = len(positions) > 0 and coverage >= min_face_coverage
        if eligible:
            prob = probabilities[positions].mean(axis=0)
            predicted = int(prob.argmax())
        else:
            prob, predicted = (np.zeros(len(labels)), -1)
        rows.append({**seg, 'prediction': labels[predicted] if predicted >= 0 else 'unknown', 'predicted_index': predicted, 'truth_index': labels.index(seg['label']), 'coverage': coverage, 'confidence': float(prob.max())})
    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError('No annotated segments for this split.')
    y, p = (out.truth_index.to_numpy(), out.predicted_index.to_numpy())
    summary = dict(accuracy=float(accuracy_score(y, p)), macro_f1=float(f1_score(y, p, labels=list(range(len(labels))), average='macro', zero_division=0)), units=len(out), unknown_units=int((p < 0).sum()), evaluable_fraction=float((p >= 0).mean()), min_face_coverage=min_face_coverage, metric_unit='image' if ACTIVE_TASK == 'vigilance' or ('metric_unit' in segments and segments.metric_unit.eq('image').all()) else 'annotated_segment', evaluation_protocols=sorted(segments.evaluation_protocol.unique()), subject_independent=bool(ACTIVE_SEGMENTS[ACTIVE_SEGMENTS.task == ACTIVE_TASK].subject_id.ne('').all()))
    if report_dir:
        report_dir = Path(report_dir)
        report_dir.mkdir(parents=True, exist_ok=True)
        out.to_csv(report_dir / 'unit_predictions.csv', index=False)
        report = classification_report(y, p, labels=list(range(len(labels))), target_names=labels, zero_division=0, output_dict=True)
        pd.DataFrame(report).T.to_csv(report_dir / 'unit_classification_report.csv')
        cm = confusion_matrix(y, p, labels=[*range(len(labels)), -1])
        pd.DataFrame(cm, index=[*labels, 'unknown'], columns=[*labels, 'unknown']).to_csv(report_dir / 'unit_confusion_matrix.csv')
        write_json(report_dir / 'unit_summary.json', summary)
    return summary

## Cell 11. Quy trình train từng nhánh

**Làm gì:** Định nghĩa cân bằng mẫu, chạy các ứng viên và lấy đúng best.pt của từng lần train.

**Đầu ra:** TaskTrainer, TaskValidator, train_task và evaluate; chưa bắt đầu train.

In [ ]:
# CELL 11: QUY TRÌNH TRAIN TỪNG NHÁNH
# Mục đích: Định nghĩa cân bằng mẫu, chạy các ứng viên và lấy đúng best.pt của từng lần train.
# Kết quả: TaskTrainer, TaskValidator, train_task và evaluate; chưa bắt đầu train.


# Đánh giá theo nhánh hiện tại, gồm các ảnh mất ROI.
class TaskValidator(DMSValidator):

    def get_dataloader(self, dataset_path, batch_size):
        return DataLoader(self.build_dataset(dataset_path), batch_size=batch_size, shuffle=False, num_workers=self.args.workers)

    def init_metrics(self, model):
        super().init_metrics(model)
        self.all_probabilities = []

    def update_metrics(self, preds, batch):
        super().update_metrics(preds, batch)
        self.all_probabilities.append(preds.detach().float().cpu().numpy())

    def get_stats(self):
        stats = super().get_stats()
        if ACTIVE_TASK != 'behavior':
            order = [str(Path(s[0]).resolve()) for s in self.dataloader.dataset.samples]
            aligned = ACTIVE_FRAMES.set_index('prepared_path').loc[order].reset_index()
            segments = ACTIVE_SEGMENTS[(ACTIVE_SEGMENTS.task == ACTIVE_TASK) & (ACTIVE_SEGMENTS.split == 'val')]
            metric = unit_scores(aligned, np.concatenate(self.all_probabilities), segments)
            stats['metrics/unit_macro_f1'] = metric['macro_f1']
            stats['fitness'] = metric['macro_f1']
        return stats

# Cân bằng lớp/nguồn/nhóm ảnh trong từng nhánh.
class TaskTrainer(DMSTrainer):

    def get_dataloader(self, dataset_path, batch_size=16, rank=-1, mode='train'):
        loader = super().get_dataloader(dataset_path, batch_size, rank, mode)
        if mode != 'train':
            return loader
        path_map = ACTIVE_FRAMES.set_index('prepared_path')
        aligned = path_map.loc[[str(Path(s[0]).resolve()) for s in loader.dataset.samples]]
        unit = 'source' if ACTIVE_TASK == 'behavior' else 'split_group' if ACTIVE_TASK == 'vigilance' else 'segment_id'
        count = aligned.groupby(['label', unit]).label.transform('size')
        n_units = aligned.groupby('label')[unit].transform('nunique')
        weights = torch.as_tensor((1 / (count * n_units)).to_numpy(copy=True), dtype=torch.double)
        generator = torch.Generator().manual_seed(self.args.seed)
        return DataLoader(loader.dataset, batch_size=batch_size, sampler=WeightedRandomSampler(weights, len(aligned), replacement=True, generator=generator), num_workers=self.args.workers, pin_memory=self.device.type == 'cuda', generator=generator)

    def get_validator(self):
        self.loss_names = ['loss']
        return TaskValidator(self.test_loader, self.save_dir, args=copy(self.args), _callbacks=self.callbacks)

# Train các ứng viên và lấy best.pt đúng từ lần chạy hiện tại.
def run_trials(dataset_root, frame, work_root, experiments, epochs=70, workers=2, seed=42, device=0, pretrained=True, extra_train_args=None):
    work_root = Path(work_root)
    rows = []
    val = frame[frame.split == 'val'].reset_index(drop=True)
    for exp in experiments:
        seed_everything(seed)
        started = time.perf_counter()
        model = YOLO(exp['model'])
        args = dict(data=str(Path(dataset_root).resolve()), trainer=DMSTrainer, epochs=epochs, patience=15, imgsz=exp['imgsz'], batch=exp['batch'], device=device, workers=workers, optimizer='AdamW', lr0=exp.get('lr0', 0.0003), lrf=0.01, weight_decay=0.0005, warmup_epochs=3, warmup_bias_lr=0.0003, cos_lr=True, dropout=0.1, seed=seed, deterministic=True, amp=torch.cuda.is_available(), cache=False, close_mosaic=0, plots=True, pretrained=pretrained, project=str(work_root / 'runs'), name=exp['name'], exist_ok=False)
        args.update(extra_train_args or {})
        model.train(**args)
        best = Path(model.trainer.best).resolve()
        if not best.is_file():
            raise FileNotFoundError(best)
        artifact = work_root / 'candidates' / exp['name']
        artifact.mkdir(parents=True, exist_ok=False)
        shutil.copy2(best, artifact / 'best.pt')
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        y, prob = predict_frame(artifact / 'best.pt', val, exp['imgsz'], device, batch=exp['batch'])
        metrics = score_predictions(y, prob, val, artifact / 'validation', bootstrap=0)
        rows.append(dict(**exp, checkpoint=str((artifact / 'best.pt').resolve()), validation_macro_f1=metrics['macro_f1'], validation_accuracy=metrics['accuracy'], train_minutes=(time.perf_counter() - started) / 60, checkpoint_mb=(artifact / 'best.pt').stat().st_size / 1024 ** 2))
        pd.DataFrame(rows).to_csv(work_root / 'validation_leaderboard.csv', index=False)
    return pd.DataFrame(rows).sort_values('validation_macro_f1', ascending=False).reset_index(drop=True)

# Huấn luyện một nhánh và tạo bảng xếp hạng validation.
def train_task(task, frames, segments, work, experiments, epochs=60, device=0, workers=2, pretrained=True, extra_train_args=None):
    activate(task, frames, segments)
    rows = []
    for exp in experiments:
        args = dict(trainer=TaskTrainer, **extra_train_args or {})
        table = run_trials(Path(work) / task / 'dataset', frames, Path(work) / task, [exp], epochs=epochs, device=device, workers=workers, pretrained=pretrained, extra_train_args=args, seed=(extra_train_args or {}).get('seed', 42))
        row = table.iloc[0].to_dict()
        row['frame_validation_macro_f1'] = row['validation_macro_f1']
        if task != 'behavior':
            val = frames[frames.split == 'val'].reset_index(drop=True)
            _, probs = predict_frame(row['checkpoint'], val, int(row['imgsz']), device)
            score = unit_scores(val, probs, segments[(segments.task == task) & (segments.split == 'val')], Path(row['checkpoint']).parent / 'validation_units')
            row['validation_macro_f1'] = score['macro_f1']
            row['validation_accuracy'] = score['accuracy']
        rows.append(dict(task=task, **row))
        pd.DataFrame(rows).to_csv(Path(work) / task / 'validation_leaderboard.csv', index=False)
    return pd.DataFrame(rows).sort_values('validation_macro_f1', ascending=False).reset_index(drop=True)

# Đánh giá model trên val hoặc test, không dùng test để chọn model.
def evaluate(task, frames, segments, model_path, imgsz, split, output, device='cpu'):
    activate(task, frames, segments)
    data = frames[frames.split == split].reset_index(drop=True)
    y, probs = predict_frame(model_path, data, imgsz, device=device)
    frame_score = score_predictions(y, probs, data, Path(output) / 'frames', bootstrap=0)
    if task == 'behavior':
        return {**frame_score, 'metric_unit': 'image'}
    segment = unit_scores(data, probs, segments[(segments.task == task) & (segments.split == split)], Path(output) / 'units')
    return {**segment, 'valid_face_macro_f1': frame_score['macro_f1']}

## Cell 12. Huấn luyện và khóa lựa chọn

**Làm gì:** Train lần lượt ba nhánh, so sánh validation rồi khóa các model sẽ kiểm tra.

**Đầu ra:** leaderboards, finalists, weights best.pt và selection_locked.json.

**Đây là cell train chính.** `balanced` so sánh 5 cấu hình, `thorough` 7 cấu hình. CPU chỉ chạy `quick_check`. Chưa dùng test để chọn model.

In [ ]:
# CELL 12: HUẤN LUYỆN VÀ KHÓA LỰA CHỌN
# Mục đích: Train lần lượt ba nhánh, so sánh validation rồi khóa các model sẽ kiểm tra.
# Kết quả: leaderboards, finalists, weights best.pt và selection_locked.json.


DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and PROFILE != 'quick_check':
    raise RuntimeError('Đã chuẩn bị dữ liệu xong nhưng chưa có GPU. Bật T4 trong Runtime → Change runtime type, '
                       'hoặc chọn PROFILE=quick_check để thử trên CPU. Full training không tự chuyển sang CPU.')
if DEVICE == 'cpu':
    NUM_WORKERS = 0
    print('Chạy thử trên CPU, tối đa 2 epochs; chưa phải kết quả huấn luyện đầy đủ.')
leaderboards = {}
for task in TASKS:
    leaderboards[task] = train_task(task, task_frames[task], unit_info, WORK, EXPERIMENTS[task], epochs=EPOCHS, workers=NUM_WORKERS, device=DEVICE, extra_train_args=dict(seed=SEED))
    display(leaderboards[task])
    backup_results()

finalists = {}
for task, table in leaderboards.items():
    table = table.assign(resource_proxy=table.checkpoint_mb * table.imgsz ** 2)
    finalists[task] = pd.DataFrame([table.sort_values('validation_macro_f1', ascending=False).iloc[0], table.sort_values('resource_proxy').iloc[0]]).drop_duplicates('name').reset_index(drop=True)
write_json(WORK / 'selection_locked.json', {k: v.to_dict('records') for k, v in finalists.items()})
print('Đã khóa model. Chưa dùng điểm test để chọn model.')

## Cell 13. Xuất ONNX và kiểm tra validation

**Làm gì:** Xuất các model đã chọn, so F1 ONNX với PyTorch rồi khóa export hợp lệ.

**Đầu ra:** ONNX, metadata/checksum và export_selection_locked.json.

Mặc định ONNX FP32 batch 1. Chấp nhận suy giảm tối đa 0,01 macro-F1; không tự bật INT8.

In [ ]:
# CELL 13: XUẤT ONNX VÀ KIỂM TRA VALIDATION
# Mục đích: Xuất các model đã chọn, so F1 ONNX với PyTorch rồi khóa export hợp lệ.
# Kết quả: ONNX, metadata/checksum và export_selection_locked.json.


# Chuẩn bị ảnh train cho calibration nếu thử INT8 sau này.
def prepare_calibration(frame, output_root, count_per_class=100, seed=42):
    """Exporters can read val; here BOTH train and val contain TRAIN images only."""
    output_root = Path(output_root)
    if output_root.exists() and any(output_root.iterdir()):
        raise FileExistsError(output_root)
    selected = pd.concat([g.sample(min(count_per_class, len(g)), random_state=seed) for _, g in frame[frame.split == 'train'].groupby('label')])
    if len(selected) < 300:
        raise ValueError('Need at least 300 representative training images for INT8 calibration.')
    for r in selected.itertuples():
        for split in ['train', 'val']:
            dest = output_root / split / r.label / Path(r.prepared_path).name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(r.prepared_path, dest)
    selected.to_csv(output_root.parent / 'calibration_manifest.csv', index=False)
    return output_root

# Xuất model và ghi checksum; bản mặc định dùng ONNX FP32.
def export_candidate(row, work_root, calibration_root=None, int8=False):
    """Use returned paths; never search other runs for an arbitrary best*.tflite."""
    from ultralytics.engine.exporter import export_formats
    artifact = Path(row['checkpoint']).parent
    model = YOLO(row['checkpoint'])
    returned = Path(model.export(format='onnx', imgsz=int(row['imgsz']), batch=1, dynamic=False, half=False, simplify=False, opset=17, device='cpu'))
    onnx_path = artifact / 'model.onnx'
    if returned.resolve() != onnx_path.resolve():
        shutil.copy2(returned, onnx_path)
    exports = {'onnx_fp32': str(onnx_path.resolve())}
    if int8:
        formats = export_formats()['Argument']
        kw = dict(imgsz=int(row['imgsz']), batch=1, device='cpu', data=str(calibration_root), fraction=1.0)
        if 'litert' in formats:
            kw.update(format='litert', quantize=8)
        else:
            kw.update(format='tflite', int8=True)
        returned = Path(model.export(**kw))
        if returned.suffix != '.tflite' or not returned.is_file():
            raise RuntimeError(f'Unexpected export output: {returned}')
        dest = artifact / 'model_int8.tflite'
        if returned.resolve() != dest.resolve():
            shutil.copy2(returned, dest)
        exports['tflite_int8'] = str(dest.resolve())
    metadata = dict(classes=CLASSES, base_size=BASE_SIZE, imgsz=int(row['imgsz']), preprocessing='PIL EXIF transpose; RGB; ImageOps.pad to base_size, bilinear, RGB padding 114; resize to imgsz bilinear; float32 /255; NCHW for ONNX', checkpoint_sha256=sha256_file(row['checkpoint']), outputs='softmax probabilities, not independently calibrated confidence', task_limit='single-image mutually exclusive four-class baseline; source confounding and unknown subject identities require external video validation', validation_macro_f1=float(row['validation_macro_f1']), artifacts={k: dict(filename=Path(v).name, sha256=sha256_file(v)) for k, v in exports.items()})
    write_json(artifact / 'metadata.json', metadata)
    return exports

# Bổ sung metadata nhãn, ROI và giới hạn của từng nhánh.
def export(task, row, frames, segments, work, int8=False):
    activate(task, frames, segments)
    calibration = None
    if int8:
        calibration = prepare_calibration(frames, Path(work) / task / ('calibration_' + row['name']), count_per_class=200)
    exports = export_candidate(row, work, calibration_root=calibration, int8=int8)
    path = Path(row['checkpoint']).parent / 'metadata.json'
    metadata = json.loads(path.read_text(encoding='utf-8'))
    metadata.update(task=task, classes=TASKS[task], roi='whole_frame' if task == 'behavior' else 'face', face_detector='YuNet 2023mar, 320x320 letterbox, confidence 0.7, margin 15%', task_limit='Independent image task outputs. Perceptual clusters do not prove identity independence. Yawn images are not temporal event annotations; validate on new people/camera.', evaluation_protocols=sorted(frames.evaluation_protocol.unique()), subject_independent=bool((frames if task == 'behavior' else segments[segments.task == task]).subject_id.ne('').all()))
    write_json(path, metadata)
    return exports

export_tables = {}
for task, table in finalists.items():
    rows = []
    for candidate in table.to_dict('records'):
        paths = export(task, candidate, task_frames[task], unit_info, WORK, int8=False)
        for fmt, path in paths.items():
            score = evaluate(task, task_frames[task], unit_info, path, int(candidate['imgsz']), 'val', Path(candidate['checkpoint']).parent / ('validation_' + fmt))
            drop = float(candidate['validation_macro_f1']) - score['macro_f1']
            rows.append(dict(candidate=candidate['name'], format=fmt, path=path, checkpoint=candidate['checkpoint'], imgsz=int(candidate['imgsz']), validation_macro_f1=score['macro_f1'], f1_drop=drop, acceptable=drop <= 0.01))
    export_tables[task] = pd.DataFrame(rows)
    display(export_tables[task])
write_json(WORK / 'export_selection_locked.json', {k: v.to_dict('records') for k, v in export_tables.items()})
for task, table in export_tables.items():
    if not table.acceptable.any():
        raise RuntimeError(f'{task}: ONNX sai lệch quá ngưỡng validation; xem báo cáo export trước khi tiếp tục.')
backup_results()

## Cell 14. Đánh giá test và chuẩn bị file Pi

**Làm gì:** Chấm toàn bộ test của các lựa chọn đã khóa, rồi chuyển sang thư mục xuất code Pi.

**Đầu ra:** final_test_results.csv, báo cáo từng nhánh và thư mục file Pi.

F1 theo ảnh, chưa phải recall sự kiện theo thời gian. Không gộp ba nhánh thành một accuracy chung. Sau cell này chạy tiếp ba cell ghi file bên dưới để hoàn tất gói Pi.

In [ ]:
# CELL 14: ĐÁNH GIÁ TEST VÀ CHUẨN BỊ FILE PI
# Mục đích: Chấm toàn bộ test của các lựa chọn đã khóa, rồi chuyển sang thư mục xuất code Pi.
# Kết quả: final_test_results.csv, báo cáo từng nhánh và thư mục file Pi.


test_rows = []
for task, table in finalists.items():
    for candidate in table.to_dict('records'):
        entries = [('pytorch_fp32', candidate['checkpoint'])]
        entries += [(r['format'], r['path']) for r in export_tables[task].to_dict('records') if r['candidate'] == candidate['name'] and r['acceptable']]
        for fmt, path in entries:
            report = WORK / task / 'reports' / candidate['name'] / fmt
            score = evaluate(task, task_frames[task], unit_info, path, int(candidate['imgsz']), 'test', report, device=DEVICE if fmt == 'pytorch_fp32' else 'cpu')
            test_rows.append(dict(task=task, candidate=candidate['name'], format=fmt, **score))
results = pd.DataFrame(test_rows)
results.to_csv(WORK / 'final_test_results.csv', index=False)
display(results)
print('F1/accuracy theo ảnh; không phải recall sự kiện buồn ngủ/ngáp theo thời gian.')
backup_results()

PREVIOUS_CWD = Path.cwd()
os.chdir(RUNTIME)

## Cell 15. Xuất roi.py cho Pi

**Làm gì:** Ghi code YuNet ROI dùng trên Raspberry Pi, cùng thuật toán với lúc chuẩn bị dữ liệu.

**Đầu ra:** File roi.py; không mở camera Colab.

In [ ]:
%%writefile roi.py
# CELL 15: XUẤT ROI.PY CHO PI
# Mục đích: Ghi code YuNet ROI dùng trên Raspberry Pi, cùng thuật toán với lúc chuẩn bị dữ liệu.
# Kết quả: File roi.py; không mở camera Colab.

"""Shared training/Pi YuNet ROI. Missing faces remain unknown, never full-frame."""
import hashlib
from pathlib import Path
import cv2

MODEL_NAME = 'face_detection_yunet_2023mar.onnx'
MODEL_SHA256 = '8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4'


class FaceCropper:
    def __init__(self):
        model = Path(__file__).with_name(MODEL_NAME)
        if not model.is_file() or hashlib.sha256(model.read_bytes()).hexdigest() != MODEL_SHA256:
            raise RuntimeError('Missing/mismatched YuNet weights. Rerun notebook setup or use the complete Pi bundle.')
        self.detector = cv2.FaceDetectorYN.create(str(model), '', (320, 320), .7, .3, 5000)

    def __call__(self, bgr):
        h, w = bgr.shape[:2]
        scale = 320 / max(h, w)
        sw, sh = max(1, round(w * scale)), max(1, round(h * scale))
        small = cv2.resize(bgr, (sw, sh))
        left_pad, top_pad = (320 - sw) // 2, (320 - sh) // 2
        square = cv2.copyMakeBorder(small, top_pad, 320 - sh - top_pad,
                                   left_pad, 320 - sw - left_pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
        _, faces = self.detector.detect(square)
        if faces is None or not len(faces):
            return None
        # Largest face is the documented single-person assumption.
        x, y, fw, fh = max(faces, key=lambda box: box[2] * box[3])[:4]
        x, y = (x - left_pad) / scale, (y - top_pad) / scale
        fw, fh = fw / scale, fh / scale
        pad_x, pad_y = fw * .15, fh * .15
        left, top = max(0, int(x - pad_x)), max(0, int(y - pad_y))
        right, bottom = min(w, int(x + fw + pad_x)), min(h, int(y + fh + pad_y))
        return bgr[top:bottom, left:right].copy() if right > left and bottom > top else None

## Cell 16. Xuất benchmark_pi.py

**Làm gì:** Ghi code preprocessing, ONNX Runtime, checksum và đo thời gian chạy model trên Pi.

**Đầu ra:** File benchmark_pi.py; chưa đo FPS Pi tại Colab.

In [ ]:
%%writefile benchmark_pi.py
# CELL 16: XUẤT BENCHMARK_PI.PY
# Mục đích: Ghi code preprocessing, ONNX Runtime, checksum và đo thời gian chạy model trên Pi.
# Kết quả: File benchmark_pi.py; chưa đo FPS Pi tại Colab.

"""Headless Raspberry Pi inference benchmark and optional camera demo.

Install: pip install numpy pillow onnxruntime
For INT8: pip install ai-edge-litert  (or a compatible tflite-runtime build)
For camera: pip install opencv-python
"""
from __future__ import annotations

import argparse
import hashlib
import json
import platform
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps


def preprocess(image, metadata):
    image = ImageOps.exif_transpose(image).convert('RGB')
    base, size = int(metadata['base_size']), int(metadata['imgsz'])
    image = ImageOps.pad(image, (base, base), method=Image.Resampling.BILINEAR,
                        color=(114, 114, 114), centering=(0.5, 0.5))
    image = image.resize((size, size), resample=Image.Resampling.BILINEAR)
    return np.asarray(image, dtype=np.float32)[None, ...] / 255.0


def affine_quantize(x, detail):
    dtype = detail['dtype']
    if np.issubdtype(dtype, np.integer):
        scale, zero = detail['quantization']
        if scale <= 0:
            raise ValueError('Missing per-tensor input quantization parameters.')
        bounds = np.iinfo(dtype)
        return np.clip(np.rint(x / scale + zero), bounds.min, bounds.max).astype(dtype)
    return x.astype(dtype)


def affine_dequantize(x, detail):
    if np.issubdtype(x.dtype, np.integer):
        scale, zero = detail['quantization']
        if scale <= 0:
            raise ValueError('Missing output quantization parameters.')
        return (x.astype(np.float32) - zero) * scale
    return x.astype(np.float32)


class Runner:
    def __init__(self, model_path, threads):
        self.path = Path(model_path)
        if self.path.suffix.lower() == '.onnx':
            import onnxruntime as ort
            options = ort.SessionOptions()
            options.intra_op_num_threads = threads
            options.inter_op_num_threads = 1
            self.session = ort.InferenceSession(str(self.path), sess_options=options,
                                                providers=['CPUExecutionProvider'])
            self.input_name = self.session.get_inputs()[0].name
            self.kind = 'onnxruntime_cpu'
        elif self.path.suffix.lower() == '.tflite':
            try:
                from ai_edge_litert.interpreter import Interpreter
            except ImportError:
                from tflite_runtime.interpreter import Interpreter
            self.interpreter = Interpreter(model_path=str(self.path), num_threads=threads)
            self.interpreter.allocate_tensors()
            self.input_detail = self.interpreter.get_input_details()[0]
            self.output_detail = self.interpreter.get_output_details()[0]
            self.kind = 'tflite_cpu'
        else:
            raise ValueError('Use a .onnx or .tflite export.')

    def __call__(self, nhwc):
        if self.kind == 'onnxruntime_cpu':
            out = self.session.run(None, {
                self.input_name: np.ascontiguousarray(nhwc.transpose(0, 3, 1, 2))})[0]
        else:
            value = nhwc
            shape = tuple(self.input_detail['shape'])
            if shape[1] == 3 and shape[-1] != 3:
                value = nhwc.transpose(0, 3, 1, 2)
            value = affine_quantize(np.ascontiguousarray(value), self.input_detail)
            self.interpreter.set_tensor(self.input_detail['index'], value)
            self.interpreter.invoke()
            out = affine_dequantize(self.interpreter.get_tensor(
                self.output_detail['index']), self.output_detail)
        return out.reshape(-1)


def verify_model(model, metadata):
    for item in metadata.get('artifacts', {}).values():
        if item['filename'] == Path(model).name:
            digest = hashlib.sha256(Path(model).read_bytes()).hexdigest()
            if digest != item['sha256']:
                raise ValueError('Model checksum differs from metadata; use the matching bundle.')
            return
    raise ValueError('Model filename is not registered in metadata.json.')


def summarize(latencies):
    return dict(p50_ms=float(np.quantile(latencies, .5) * 1000),
                p95_ms=float(np.quantile(latencies, .95) * 1000),
                mean_ms=float(np.mean(latencies) * 1000))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', required=True)
    parser.add_argument('--metadata', required=True)
    parser.add_argument('--image', help='A real driver image for repeated-image benchmark')
    parser.add_argument('--camera', type=int, help='OpenCV camera index; omit to benchmark an image')
    parser.add_argument('--seconds', type=float, default=60)
    parser.add_argument('--iterations', type=int, default=200)
    parser.add_argument('--warmup', type=int, default=20)
    parser.add_argument('--threads', type=int, default=4)
    parser.add_argument('--show', action='store_true', help='Show camera preview on a desktop')
    parser.add_argument('--output', default='pi_benchmark.json')
    args = parser.parse_args()
    if args.threads < 1 or args.iterations < 1 or args.warmup < 0 or args.seconds <= 0:
        parser.error('threads, iterations and seconds must be positive; warmup cannot be negative.')
    if args.camera is None and not args.image:
        parser.error('Supply --image or --camera.')
    metadata = json.loads(Path(args.metadata).read_text(encoding='utf-8'))
    verify_model(args.model, metadata)
    runner = Runner(args.model, args.threads)
    common = dict(hardware=platform.platform(), machine=platform.machine(),
                  runtime=runner.kind, threads=args.threads, model=str(Path(args.model).resolve()),
                  model_size_mb=Path(args.model).stat().st_size / 1024**2,
                  input_size=metadata['imgsz'], warmup=args.warmup)
    if args.camera is None:
        with Image.open(args.image) as im:
            original = im.copy()
        value = preprocess(original, metadata)
        for _ in range(args.warmup):
            runner(value)
        inference, pipeline = [], []
        for _ in range(args.iterations):
            start = time.perf_counter()
            x = preprocess(original, metadata)
            after_preprocess = time.perf_counter()
            probabilities = runner(x)
            end = time.perf_counter()
            inference.append(end - after_preprocess)
            pipeline.append(end - start)
        label = metadata['classes'][int(probabilities.argmax())]
        report = dict(**common, mode='repeated_image_no_camera_or_disk_io',
            iterations=args.iterations, inference=summarize(inference),
            preprocess_plus_inference=summarize(pipeline),
            serial_pipeline_fps_estimate=1 / float(np.mean(pipeline)), prediction=label,
            confidence=float(probabilities.max()))
    else:
        import cv2
        cap = cv2.VideoCapture(args.camera)
        if not cap.isOpened():
            raise RuntimeError('Camera could not be opened. Picamera2 may need a separate adapter.')
        try:
            # Exclude warmup, camera opening, and initial exposure adjustment from timings.
            for _ in range(args.warmup):
                ok, frame = cap.read()
                if not ok:
                    raise RuntimeError('Camera read failed during warmup.')
                runner(preprocess(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), metadata))
            started = time.perf_counter()
            inference, pipeline, n = [], [], 0
            while time.perf_counter() - started < args.seconds:
                start = time.perf_counter()
                ok, frame = cap.read()
                if not ok:
                    raise RuntimeError('Camera read failed.')
                x = preprocess(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), metadata)
                infer_start = time.perf_counter()
                probabilities = runner(x)
                inference.append(time.perf_counter() - infer_start)
                if args.show:
                    label = metadata['classes'][int(probabilities.argmax())]
                    cv2.putText(frame, f'{label} {probabilities.max():.2f}', (15, 35),
                                cv2.FONT_HERSHEY_SIMPLEX, .8, (0, 255, 0), 2)
                    cv2.imshow('DMS frame classifier', frame)
                    if cv2.waitKey(1) & 0xff == ord('q'):
                        break
                pipeline.append(time.perf_counter() - start)
                n += 1
            elapsed = time.perf_counter() - started
            if not pipeline:
                raise RuntimeError('No camera frames were measured.')
            report = dict(**common, mode='camera_pipeline', preview=args.show,
                measured_frames=n, elapsed_seconds=elapsed, measured_fps=n / elapsed,
                inference=summarize(inference), capture_preprocess_inference=summarize(pipeline))
        finally:
            cap.release()
            if args.show:
                cv2.destroyAllWindows()
    Path(args.output).write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))


if __name__ == '__main__':
    main()

## Cell 17. Xuất demo_pi_multitask.py

**Làm gì:** Ghi demo camera ba nhánh, lọc xác suất theo thời gian và xử lý mất mặt.

**Đầu ra:** File demo_pi_multitask.py; chưa chạy camera.

In [ ]:
%%writefile demo_pi_multitask.py
# CELL 17: XUẤT DEMO_PI_MULTITASK.PY
# Mục đích: Ghi demo camera ba nhánh, lọc xác suất theo thời gian và xử lý mất mặt.
# Kết quả: File demo_pi_multitask.py; chưa chạy camera.

"""Three task ONNX demo; independent outputs and bounded inference frequencies."""
from __future__ import annotations
import argparse
from collections import deque
import json
from pathlib import Path
import time
import cv2
import numpy as np
from PIL import Image
from benchmark_pi import Runner, preprocess, verify_model, summarize
from roi import FaceCropper


class TemporalDecision:
    def __init__(self, window_s=5., confirmation_s=2., confidence=.65):
        self.window_s, self.confirmation_s, self.confidence = window_s, confirmation_s, confidence
        self.history = deque()
        self.pending, self.pending_since = None, None

    def update(self, timestamp, probability, classes):
        if probability is None:
            # Missing face cannot retain a stale normal or drowsy decision.
            self.history.clear()
            self.pending, self.pending_since = None, None
            return dict(label='unknown', confirmed=False, confidence=None)
        self.history.append((timestamp, np.asarray(probability)))
        while self.history and timestamp - self.history[0][0] > self.window_s:
            self.history.popleft()
        average = np.mean([p for _, p in self.history], axis=0)
        value = classes[int(average.argmax())]
        if float(average.max()) < self.confidence:
            self.pending, self.pending_since = None, None
            return dict(label='unknown', confirmed=False, confidence=float(average.max()))
        if self.pending != value:
            self.pending, self.pending_since = value, timestamp
        return dict(label=value, confirmed=timestamp - self.pending_since >= self.confirmation_s,
                    confidence=float(average.max()))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--camera', type=int, default=0)
    parser.add_argument('--video', help='Optional prerecorded file instead of a live camera')
    parser.add_argument('--seconds', type=float, default=60)
    parser.add_argument('--threads', type=int, default=4)
    parser.add_argument('--show', action='store_true')
    parser.add_argument('--output', default='multitask_pi_benchmark.json')
    args = parser.parse_args()
    if args.seconds <= 0 or args.threads < 1:
        parser.error('seconds and threads must be positive.')
    config_path = Path(args.config).resolve()
    root = config_path.parent
    config = json.loads(config_path.read_text(encoding='utf-8'))
    runners, metadata, last_run, latencies, states, decisions = {}, {}, {}, {}, {}, {}
    for task, item in config['tasks'].items():
        model = root / item['model']
        metadata[task] = json.loads((root / item['metadata']).read_text(encoding='utf-8'))
        verify_model(model, metadata[task])
        runners[task] = Runner(model, args.threads)
        last_run[task], latencies[task] = -float('inf'), []
        states[task] = dict(label='unknown', confirmed=False, confidence=None)
        decisions[task] = TemporalDecision(window_s=item.get('window_s', 1.),
            confirmation_s=item.get('confirmation_s', .5), confidence=item.get('confidence', .65))
    cropper = FaceCropper()
    cap = cv2.VideoCapture(args.video if args.video else args.camera)
    if not cap.isOpened():
        raise RuntimeError('Cannot open video source; CSI/Picamera2 may need an adapter.')
    loops, loop_latencies, detection_latencies = 0, [], []
    try:
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError('First frame could not be read.')
        # Fixed-shape dummy warmup is not counted in the measured camera FPS.
        for task, runner in runners.items():
            size = metadata[task]['imgsz']
            for _ in range(5):
                runner(np.zeros((1, size, size, 3), dtype=np.float32))
        started, last_log = time.perf_counter(), -float('inf')
        while time.perf_counter() - started < args.seconds:
            loop_start = time.perf_counter()
            ok, frame = cap.read()
            if not ok:
                break
            now = loop_start - started
            # A prerecorded video's temporal decisions use source timestamps, not decode speed.
            timeline = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000 if args.video else now
            due = [task for task, item in config['tasks'].items()
                   if timeline - last_run[task] >= 1 / item['hz']]
            face = None
            if any(metadata[t]['roi'] == 'face' for t in due):
                detect_start = time.perf_counter()
                face = cropper(frame)
                detection_latencies.append(time.perf_counter() - detect_start)
            for task in due:
                start = time.perf_counter()
                roi = frame if metadata[task]['roi'] == 'whole_frame' else face
                probability = None
                if roi is not None:
                    image = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
                    probability = runners[task](preprocess(image, metadata[task]))
                states[task] = decisions[task].update(timeline, probability, metadata[task]['classes'])
                last_run[task] = timeline
                latencies[task].append(time.perf_counter() - start)
            if now - last_log >= 1:
                print(json.dumps(dict(time_s=round(timeline, 2), states=states)))
                last_log = now
            if args.show:
                for i, (task, state) in enumerate(states.items()):
                    text = f"{task}: {state['label']} confirmed={state['confirmed']}"
                    cv2.putText(frame, text, (12, 30 + i * 30), cv2.FONT_HERSHEY_SIMPLEX,
                                .65, (0, 255, 0), 2)
                cv2.imshow('DMS three tasks', frame)
                if cv2.waitKey(1) & 0xff == ord('q'):
                    break
            loops += 1
            loop_latencies.append(time.perf_counter() - loop_start)
        elapsed = time.perf_counter() - started
    finally:
        cap.release()
        if args.show:
            cv2.destroyAllWindows()
    if not loops:
        raise RuntimeError('No frames were measured.')
    report = dict(mode='prerecorded_decode' if args.video else 'live_camera',
        threads=args.threads, elapsed_s=elapsed, captured_frames=loops,
        camera_loop_fps=loops / elapsed, camera_loop=summarize(loop_latencies),
        face_detection=summarize(detection_latencies) if detection_latencies else None,
        tasks={task: dict(updates=len(values), updates_per_wall_second=len(values) / elapsed,
                         preprocess_plus_inference=summarize(values) if values else None)
               for task, values in latencies.items()},
        note='Camera-loop FPS is not per-model inference FPS. Scheduling/thresholds '
             'are demo settings, not calibrated event-detection performance.')
    Path(args.output).write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))


if __name__ == '__main__':
    main()

## Cell 18. Đóng gói và tải kết quả

**Làm gì:** Tạo deployment JSON, kèm weights, báo cáo, code Pi và license vào ZIP.

**Đầu ra:** dms_v6_visible_results.zip; lưu Drive nếu đã bật.

ZIP không chứa ảnh dataset. Hai cấu hình quality/resource chọn bằng validation; proxy tài nguyên không thay cho FPS Pi thật.

In [ ]:
# CELL 18: ĐÓNG GÓI VÀ TẢI KẾT QUẢ
# Mục đích: Tạo deployment JSON, kèm weights, báo cáo, code Pi và license vào ZIP.
# Kết quả: dms_v6_visible_results.zip; lưu Drive nếu đã bật.


os.chdir(PREVIOUS_CWD)

# Tạo hai cấu hình triển khai và ZIP không chứa ảnh dataset.
def make_bundle(work, selections, runtime, max_drop=0.01):
    """Lock export choices on validation before any test evaluation; package portable Pi paths."""
    work, runtime = (Path(work), Path(runtime))
    export_rows = []
    for task, table in selections.items():
        accepted = table[(table.format == 'onnx_fp32') & table.acceptable]
        if accepted.empty:
            raise ValueError(f'{task}: no ONNX export within {max_drop:.3f} validation F1 drop.')
        export_rows.append(accepted.assign(task=task))
    exports = pd.concat(export_rows, ignore_index=True)
    registered_onnx = {Path(path).resolve() for path in exports.path}
    for mode in ['quality', 'resource']:
        config = dict(tasks={}, note='Demo temporal thresholds require labelled video validation.')
        for task, group in exports.groupby('task'):
            group = group.copy()
            group['resource_proxy'] = group.path.map(lambda p: Path(p).stat().st_size) * group.imgsz ** 2
            key, ascending = ('validation_macro_f1', False) if mode == 'quality' else ('resource_proxy', True)
            row = group.sort_values(key, ascending=ascending).iloc[0]
            metadata = Path(row.checkpoint).parent / 'metadata.json'
            config['tasks'][task] = dict(model=Path(row.path).relative_to(work).as_posix(), metadata=metadata.relative_to(work).as_posix(), hz=4 if task == 'vigilance' else 2, window_s=5 if task == 'vigilance' else 1, confirmation_s=2 if task == 'vigilance' else 0.5, confidence=0.65)
        write_json(work / f'deployment_{mode}.json', config)
    import shutil
    for name in ['demo_pi_multitask.py', 'benchmark_pi.py', 'roi.py', 'face_detection_yunet_2023mar.onnx', 'YUNET_LICENSE.txt']:
        shutil.copy2(runtime / name, work / name)
    pi_readme = 'Raspberry Pi OS 64-bit\n\npython -m pip install numpy pillow opencv-python onnxruntime\n\npython demo_pi_multitask.py --config deployment_resource.json --camera 0 --seconds 60 --output resource_pi.json\n\nModels use three independent image-label tasks. Event thresholds are demo defaults.\n\nRun from this directory. Measure on the actual Pi; Colab FPS is not Pi FPS.\n'
    (work / 'PI_README.txt').write_text(pi_readme, encoding='utf-8')
    bundle = work / 'dms_v6_visible_results.zip'
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for path in sorted(work.rglob('*')):
            if not path.is_file() or path == bundle:
                continue
            rel = path.relative_to(work)
            if any((part in {'dataset', 'face_frames', 'runtime', '__pycache__', '.yolo_config'} for part in rel.parts)):
                continue
            if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                continue
            if path.suffix.lower() == '.onnx' and path.name != 'face_detection_yunet_2023mar.onnx' and (path.resolve() not in registered_onnx):
                continue
            if 'runs' in rel.parts and path.suffix.lower() not in {'.csv', '.yaml'}:
                continue
            z.write(path, rel)
    return bundle

bundle = make_bundle(WORK, export_tables, RUNTIME)
print('ZIP kết quả:', bundle)
print('Kích thước: %.1f MB' % (bundle.stat().st_size / 1024 ** 2))
backup_results()
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))

## Sau khi chạy xong

Giải nén ZIP trên Raspberry Pi OS 64-bit, mở terminal tại thư mục kết quả:

```bash
python -m pip install numpy pillow opencv-python onnxruntime
python demo_pi_multitask.py --config deployment_resource.json --camera 0 --seconds 60 --output resource_pi.json
```

Thử `deployment_quality.json` để so sánh; thêm `--show` nếu có màn hình.
CSI/Picamera2 có thể cần adapter. Ngưỡng xác nhận demo cần validation video có nhãn
sự kiện. Đo FPS trên Pi thật, không lấy FPS Colab làm FPS Pi.

**Nội dung báo cáo:** nguồn và số ảnh/lớp; cách chia và giới hạn ID người; ảnh lỗi/trùng,
ROI coverage; cấu hình train; bảng validation; F1/accuracy/confusion matrix test từng nhánh;
sai lệch ONNX; kích thước model và độ trễ/FPS Pi thật. `quick_check` chỉ kiểm tra code.

Notebook chưa chứa weights classifier đã train đầy đủ trên dữ liệu thật. Kiểm thử dữ liệu
giả chỉ xác nhận pipeline chạy. Không mô phỏng buồn ngủ/dùng điện thoại khi đang lái xe.

Nguồn: [State Farm](https://www.kaggle.com/datasets/rightway11/state-farm-distracted-driver-detection),
[DDD](https://www.kaggle.com/datasets/ismailnasri20/driver-drowsiness-dataset-ddd),
[Yawn Eye](https://www.kaggle.com/datasets/serenaraju/yawn-eye-dataset-new),
[YuNet](https://github.com/opencv/opencv_zoo/tree/main/models/face_detection_yunet),
[Ultralytics](https://docs.ultralytics.com/tasks/classify/).
Quyền sử dụng dữ liệu gốc vẫn áp dụng; MIT license YuNet có trong ZIP.
